# 第 1 讲：总览与 Tokenization

这是 CS336 Lecture 01 的**完整可执行学习版**。

目标：你只看这个 notebook，也能按第一讲的结构和节奏学习；Markdown 文件只作为纯文本备份和快速复习。

来源：

- 官方讲义源码：`external/cs336-lectures/lecture_01.py`
- 中文 Markdown 版：`notes/lectures/lecture-01-overview-tokenization.zh.md`
- 英文 Markdown 版：`notes/lectures/lecture-01-overview-tokenization.en.md`

学习方法：

1. 从上往下读，不要跳到 BPE 代码。
2. 每个代码 cell 先直接运行，再改一两个字符串观察输出。
3. 遇到术语时，优先记英文：`tokenization`、`vocabulary`、`compression ratio`、`BPE`、`training loop`。


## 0. 本讲路线图

官方第一讲的节奏大致是：

```text
为什么有 CS336
-> frontier model 为什么不能完整复现
-> 哪些知识能迁移
-> 语言模型发展脉络
-> 课程作业结构
-> tokenization
-> character / byte / word / BPE
-> 下一讲 resource accounting
```

你今天不需要掌握所有历史和论文名，但要抓住两条主线：

- CS336 不是教你调 API，而是教你理解 API 下面的语言模型训练栈。
- tokenization 是第一步：模型必须先把文本变成整数序列，才能进入 embedding 和 Transformer。


## 1. 为什么要有这门课

研究者和工程师这些年一直在往更高的抽象层走：

- 2016 年左右，很多研究者会自己实现并训练模型。
- 2018 年左右，很多人下载 BERT 之类的预训练模型，然后 fine-tune。
- 今天，很多人直接 prompt GPT、Claude、Gemini 这类 API 模型。

抽象层提高了效率，但语言模型这层抽象并不严密。你想做真正底层、可靠、可迁移的工作，就必须理解 API 下面那套东西：

```text
tokenization -> architecture -> training -> systems -> data -> evaluation -> alignment
```

这门课的核心方法是：**通过亲手构建来理解**。


## 2. Frontier Model 不可能在课上复现

最前沿的大模型训练成本太高，个人学习仓库或课堂都不可能完整复现。

关键问题不是：

```text
我们能不能从零训练 GPT-4？
```

更有价值的问题是：

```text
哪些从小模型里学到的知识，可以迁移到 frontier-scale model？
```

第一讲把可迁移知识分成三类：

- **Mechanics**：机制。比如 Transformer、tokenizer、optimizer、并行训练、训练循环怎么工作。
- **Mindset**：思维方式。把 compute、memory、bandwidth、data 都当成稀缺资源。
- **Intuitions**：经验直觉。比如什么数据好、什么模型设计好。这类东西跨尺度迁移不一定稳定。


## 3. 主线：效率

CS336 会反复回到一个问题：

```text
在固定 compute、memory、communication、data 约束下，怎么训练出最好的模型？
```

所谓 bitter lesson 不是“只要规模够大，算法不重要”。更准确的理解是：

```text
能随规模一起扩展的算法才重要。
```

规模越大，浪费越贵。所以模型结构、kernel、数据过滤、batching、超参数选择，本质上都是效率问题。

下面这个小 cell 不涉及深度学习，只是把“固定资源下做权衡”的感觉跑出来。


In [1]:
# 一个玩具版 resource budget：总预算固定时，模型大小和数据量需要权衡。
# 这里只是建立直觉，不是 CS336 的正式 scaling law。

compute_budget = 1_000_000
choices = [
    {"model_size": 10, "tokens": 100_000},
    {"model_size": 100, "tokens": 10_000},
    {"model_size": 1_000, "tokens": 1_000},
]

for c in choices:
    cost = c["model_size"] * c["tokens"]
    print(c, "cost=", cost, "fits_budget=", cost <= compute_budget)


{'model_size': 10, 'tokens': 100000} cost= 1000000 fits_budget= True
{'model_size': 100, 'tokens': 10000} cost= 1000000 fits_budget= True
{'model_size': 1000, 'tokens': 1000} cost= 1000000 fits_budget= True


## 4. 语言模型的简史

神经网络之前：

- Shannon 用语言模型来研究英语的信息熵。
- N-gram 模型曾用于机器翻译和语音识别。

神经网络阶段：

- LSTM 和早期 neural language model 开始用学习到的模型处理序列。
- Seq2seq 推动了机器翻译。
- Attention 和 Transformer 成为现代语言模型的基础。
- Adam 这类优化器让深层模型训练更可行。

Scaling 阶段：

- GPT-2 展示了流畅生成和早期 zero-shot 能力。
- Scaling laws 让模型性能随规模变化更可预测。
- GPT-3 展示了大规模下的 in-context learning。
- Chinchilla 一类工作强调模型大小和数据量之间的 compute-optimal 权衡。

开放模型阶段：

- 开放权重和开放训练项目让我们可以研究现代语言模型的做法。
- CS336 正是利用这些公开资料来讲清语言模型底层栈。


## 5. 什么是语言模型

一句话版本：

```text
语言模型是在 token 序列上定义概率分布的模型。
```

它的使用界面在变：

- 2018：一个拿来 fine-tune 的模型。
- 2020：一个拿来 prompt 的模型。
- 2022：一个可以 chat 的模型。
- 2026：一个可以作为 agent 自主行动的模型。

但底层基础没有变：

```text
tokens + attention + optimization + kernels + data + evaluation
```


## 6. 课程大纲：CS336 要你从哪里搭起来

Assignment 1：Basics

- Tokenization
- Transformer architecture
- Cross-entropy loss
- AdamW optimizer
- Training loop
- Resource accounting

Assignment 2：Systems

- Kernels
- GPU efficiency
- Parallelism
- Inference

Assignment 3：Scaling Laws

- 预测模型大小、数据量、compute 改变时，性能会怎么变。

Assignment 4：Data

- Evaluation
- Curation
- Filtering
- Deduplication
- Data mixing

Assignment 5：Alignment

- Supervised fine-tuning
- Preference learning
- RLHF 相关算法和系统

今天这份 notebook 聚焦 Assignment 1 的第一个入口：**tokenization**。


## 7. Tokenization：第一单元

Tokenization 要回答的问题是：

```text
模型操作的基本单位是什么？
```

形式上，tokenizer 在原始文本和整数 token 序列之间转换：

- `encode`: string -> list[int]
- `decode`: list[int] -> string

语言模型通常不直接处理 Python 字符串。它处理整数 id，然后通过 embedding table 把这些 id 映射成向量。

先定义几个后面都会用到的小工具。


In [2]:
from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass


def compression_ratio(text: str, token_ids: list[int]) -> float:
    """UTF-8 bytes per token. Higher usually means fewer tokens for the same text."""
    return len(text.encode("utf-8")) / len(token_ids)


def show_tokenization(name: str, text: str, token_ids: list[int], decoded: str) -> None:
    print(f"[{name}]")
    print("text:       ", repr(text))
    print("token ids:  ", token_ids)
    print("decoded:    ", repr(decoded))
    print("round trip: ", decoded == text)
    print("num bytes:  ", len(text.encode("utf-8")))
    print("num tokens: ", len(token_ids))
    print("ratio:      ", compression_ratio(text, token_ids))


sample = "Hello, 🌍! 你好!"
print(sample)
print("utf8 bytes:", list(sample.encode("utf-8")))
print("num bytes:", len(sample.encode("utf-8")))


Hello, 🌍! 你好!
utf8 bytes: [72, 101, 108, 108, 111, 44, 32, 240, 159, 140, 141, 33, 32, 228, 189, 160, 229, 165, 189, 33]
num bytes: 20


## 8. Character Tokenization

最直接的方法：把每个 Unicode 字符变成一个整数 code point。

- `ord(char)`：字符 -> 整数
- `chr(number)`：整数 -> 字符

优点：

- 简单。
- 可以把文本 encode 后再 decode 回来。

缺点：

- Unicode 字符数量很大。
- 稀有字符会浪费 vocabulary 容量。
- 相比现代 tokenizer，压缩率差。


In [3]:
class CharacterTokenizer:
    def encode(self, text: str) -> list[int]:
        return [ord(ch) for ch in text]

    def decode(self, token_ids: list[int]) -> str:
        return "".join(chr(i) for i in token_ids)


char_tokenizer = CharacterTokenizer()
char_ids = char_tokenizer.encode(sample)
show_tokenization("character", sample, char_ids, char_tokenizer.decode(char_ids))

print("ord('a') =", ord("a"))
print("chr(97) =", chr(97))
print("ord('🌍') =", ord("🌍"))


[character]
text:        'Hello, 🌍! 你好!'
token ids:   [72, 101, 108, 108, 111, 44, 32, 127757, 33, 32, 20320, 22909, 33]
decoded:     'Hello, 🌍! 你好!'
round trip:  True
num bytes:   20
num tokens:  13
ratio:       1.5384615384615385
ord('a') = 97
chr(97) = a
ord('🌍') = 127757


## 9. Byte Tokenization

另一种方法：先把字符串编码成 UTF-8 bytes，再把每个 byte 当成一个 token。

优点：

- 固定小词表：256 个 byte 值。
- 可以表示任何 UTF-8 文本。

缺点：

- 压缩率差：基本是一 byte 一个 token。
- 序列会变长。
- 标准 Transformer attention 的成本随序列长度平方增长，所以长序列很贵。


In [4]:
class ByteTokenizer:
    def encode(self, text: str) -> list[int]:
        return list(text.encode("utf-8"))

    def decode(self, token_ids: list[int]) -> str:
        return bytes(token_ids).decode("utf-8")


byte_tokenizer = ByteTokenizer()
byte_ids = byte_tokenizer.encode(sample)
show_tokenization("byte", sample, byte_ids, byte_tokenizer.decode(byte_ids))

for ch in ["a", "🌍", "你"]:
    print(repr(ch), "utf8 bytes=", list(ch.encode("utf-8")))


[byte]
text:        'Hello, 🌍! 你好!'
token ids:   [72, 101, 108, 108, 111, 44, 32, 240, 159, 140, 141, 33, 32, 228, 189, 160, 229, 165, 189, 33]
decoded:     'Hello, 🌍! 你好!'
round trip:  True
num bytes:   20
num tokens:  20
ratio:       1.0
'a' utf8 bytes= [97]
'🌍' utf8 bytes= [240, 159, 140, 141]
'你' utf8 bytes= [228, 189, 160]


## 10. Word Tokenization

传统 NLP 常常把文本切成词或类似词的片段。

优点：

- token 对人来说有意义。
- 压缩率通常不错。

缺点：

- vocabulary 可能巨大。
- 稀有词很难学。
- 没见过的新词需要 unknown token 机制。
- 固定 vocabulary size 不自然。

下面这个 toy tokenizer 只是演示“按词切分”的问题，不是现代 LLM tokenizer。


In [5]:
def simple_word_chunks(text: str) -> list[str]:
    # Keep word-like spans together, keep punctuation/space as separate chunks.
    return re.findall(r"\w+|[^\w]", text)

texts = [
    "I'll say supercalifragilisticexpialidocious!",
    "hello hello",
    "模型会把文本切成 token",
]

for text in texts:
    chunks = simple_word_chunks(text)
    print("text:  ", repr(text))
    print("chunks:", chunks)
    print("num chunks:", len(chunks))
    print()


text:   "I'll say supercalifragilisticexpialidocious!"
chunks: ['I', "'", 'll', ' ', 'say', ' ', 'supercalifragilisticexpialidocious', '!']
num chunks: 8

text:   'hello hello'
chunks: ['hello', ' ', 'hello']
num chunks: 3

text:   '模型会把文本切成 token'
chunks: ['模型会把文本切成', ' ', 'token']
num chunks: 3



## 11. 现代 tokenizer：用 tiktoken 观察

`tiktoken` 是 OpenAI 的 tokenizer 库。这里用它观察真实 tokenizer 的行为。

重点不是记住某个 token id，而是观察这些现象：

- 空格经常和后面的词合并到同一个 token。
- 同一个词在句首和句中可能 token 不同。
- 数字会按几位一组切开。
- 中英文、emoji、标点的 token 数量差异很大。


In [6]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")
modern_ids = encoding.encode(sample)
show_tokenization("tiktoken o200k_base", sample, modern_ids, encoding.decode(modern_ids))

examples = [
    "hello",
    " hello",
    "hello hello",
    "1234567890",
    "模型会把文本切成 token",
    "function_call(arguments={\"x\": 1})",
]

for text in examples:
    ids = encoding.encode(text)
    print(repr(text), "->", ids, "num_tokens=", len(ids), "ratio=", compression_ratio(text, ids))


[tiktoken o200k_base]
text:        'Hello, 🌍! 你好!'
token ids:   [13225, 11, 130321, 235, 0, 220, 177519, 0]
decoded:     'Hello, 🌍! 你好!'
round trip:  True
num bytes:   20
num tokens:  8
ratio:       2.5
'hello' -> [24912] num_tokens= 1 ratio= 5.0
' hello' -> [40617] num_tokens= 1 ratio= 6.0
'hello hello' -> [24912, 40617] num_tokens= 2 ratio= 5.5
'1234567890' -> [7633, 19354, 29338, 15] num_tokens= 4 ratio= 2.5
'模型会把文本切成 token' -> [184232, 6016, 32180, 145683, 34921, 5543, 6602] num_tokens= 7 ratio= 4.285714285714286
'function_call(arguments={"x": 1})' -> [2706, 25158, 64973, 27642, 87, 1243, 220, 16, 9263] num_tokens= 9 ratio= 3.6666666666666665


## 12. Vocabulary Size 与 Compression Ratio

这里有两个相互拉扯的因素：

- vocabulary 越大，常见片段越可能变成单 token，压缩率可能更好。
- vocabulary 越大，模型的 embedding 层和输出层也越大，而且更稀疏。

第一讲里的 compression ratio 指：

```text
UTF-8 byte 数量 / token 数量
```

ratio 越高，表示同样文本需要的 token 越少，通常对 attention 成本更友好。


In [7]:
comparisons = []
for name, tokenizer in [
    ("character", char_tokenizer),
    ("byte", byte_tokenizer),
]:
    ids = tokenizer.encode(sample)
    comparisons.append((name, len(ids), compression_ratio(sample, ids)))

ids = encoding.encode(sample)
comparisons.append(("tiktoken", len(ids), compression_ratio(sample, ids)))

print("text:", repr(sample))
print("bytes:", len(sample.encode("utf-8")))
for name, num_tokens, ratio in comparisons:
    print(f"{name:10s} tokens={num_tokens:2d} ratio={ratio:.3f}")


text: 'Hello, 🌍! 你好!'
bytes: 20
character  tokens=13 ratio=1.538
byte       tokens=20 ratio=1.000
tiktoken   tokens= 8 ratio=2.500


## 13. Byte Pair Encoding：核心直觉

Byte Pair Encoding，也就是 BPE，是 bytes 和 words 之间的折中方案。

基本思路：

1. 一开始把每个 byte 当成 token。
2. 统计训练文本里相邻 token pair 的频率。
3. 把最常见的相邻 pair 合并成一个新 token。
4. 重复，直到 vocabulary 达到目标大小。

直觉：

- 常见 byte 序列会变成单个 token。
- 罕见序列仍然拆成更小的片段。
- tokenizer 学到的是适配训练数据的 vocabulary。

先看 BPE 最核心的一个操作：`merge`。


In [8]:
def count_adjacent_pairs(token_ids: list[int]) -> dict[tuple[int, int], int]:
    counts = defaultdict(int)
    for left, right in zip(token_ids, token_ids[1:]):
        counts[(left, right)] += 1
    return dict(counts)


def merge(token_ids: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    merged = []
    i = 0
    while i < len(token_ids):
        if i + 1 < len(token_ids) and (token_ids[i], token_ids[i + 1]) == pair:
            merged.append(new_id)
            i += 2
        else:
            merged.append(token_ids[i])
            i += 1
    return merged


toy = "the cat in the hat"
toy_ids = list(toy.encode("utf-8"))
counts = count_adjacent_pairs(toy_ids)
most_common_pair = max(counts, key=counts.get)

print("text:", toy)
print("byte ids:", toy_ids)
print("most common pair:", most_common_pair, "count=", counts[most_common_pair])
print("as bytes:", bytes(most_common_pair))
print("after merge:", merge(toy_ids, most_common_pair, 256))


text: the cat in the hat
byte ids: [116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]
most common pair: (116, 104) count= 2
as bytes: b'th'
after merge: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]


## 14. 训练一个极简 BPE tokenizer

下面这个版本故意写得很慢，但逻辑清楚：

1. 初始 vocab 是 256 个 byte。
2. 每一轮统计相邻 pair。
3. 找到最常见 pair。
4. 分配一个新的 token id。
5. 把这个 pair 合并。

这对应官方第一讲里的 toy BPE 实现。


In [9]:
@dataclass(frozen=True)
class BPEParams:
    vocab: dict[int, bytes]
    merges: dict[tuple[int, int], int]


def train_tiny_bpe(text: str, num_merges: int) -> BPEParams:
    token_ids = list(text.encode("utf-8"))
    vocab = {i: bytes([i]) for i in range(256)}
    merges = {}

    print("training text:", repr(text))
    print("start:", token_ids, "num_tokens=", len(token_ids))
    for step in range(num_merges):
        counts = count_adjacent_pairs(token_ids)
        pair = max(counts, key=counts.get)
        new_id = 256 + step
        merges[pair] = new_id
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        token_ids = merge(token_ids, pair, new_id)
        print(f"merge {step + 1}: {pair} -> {new_id}, bytes={vocab[new_id]!r}, num_tokens={len(token_ids)}")

    print("final ids:", token_ids)
    print("compression ratio:", compression_ratio(text, token_ids))
    return BPEParams(vocab=vocab, merges=merges)


params = train_tiny_bpe("the cat in the hat", num_merges=5)


training text: 'the cat in the hat'
start: [116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116] num_tokens= 18
merge 1: (116, 104) -> 256, bytes=b'th', num_tokens=16
merge 2: (256, 101) -> 257, bytes=b'the', num_tokens=14
merge 3: (257, 32) -> 258, bytes=b'the ', num_tokens=12
merge 4: (97, 116) -> 259, bytes=b'at', num_tokens=10
merge 5: (258, 99) -> 260, bytes=b'the c', num_tokens=9
final ids: [260, 259, 32, 105, 110, 32, 258, 104, 259]
compression ratio: 2.0


## 15. 使用训练好的 BPE tokenizer

训练 tokenizer 和使用 tokenizer 是两件事：

- **训练**：从语料里学 vocabulary 和 merges。
- **使用**：拿固定的 vocabulary 和 merges 去 encode 新文本。

Assignment 1 会要求你实现更完整、更快的版本，包括 special token、pre-tokenization、性能优化等。


In [10]:
class TinyBPETokenizer:
    def __init__(self, params: BPEParams) -> None:
        self.params = params

    def encode(self, text: str) -> list[int]:
        token_ids = list(text.encode("utf-8"))
        for pair, new_id in self.params.merges.items():
            token_ids = merge(token_ids, pair, new_id)
        return token_ids

    def decode(self, token_ids: list[int]) -> str:
        pieces = [self.params.vocab[i] for i in token_ids]
        return b"".join(pieces).decode("utf-8")


tiny_bpe = TinyBPETokenizer(params)
for text in ["the cat in the hat", "the quick brown fox", "hat hat hat"]:
    ids = tiny_bpe.encode(text)
    print("text:   ", repr(text))
    print("ids:    ", ids)
    print("decoded:", repr(tiny_bpe.decode(ids)))
    print("ratio:  ", compression_ratio(text, ids))
    print()


text:    'the cat in the hat'
ids:     [260, 259, 32, 105, 110, 32, 258, 104, 259]
decoded: 'the cat in the hat'
ratio:   2.0

text:    'the quick brown fox'
ids:     [258, 113, 117, 105, 99, 107, 32, 98, 114, 111, 119, 110, 32, 102, 111, 120]
decoded: 'the quick brown fox'
ratio:   1.1875

text:    'hat hat hat'
ids:     [104, 259, 32, 104, 259, 32, 104, 259]
decoded: 'hat hat hat'
ratio:   1.375



## 16. Assignment 1 预览：你之后要实现什么

Assignment 1 会让你实现：

- BPE tokenizer
- Transformer
- Cross-entropy loss
- AdamW
- Training loop
- Resource accounting

要一直平衡三件事：

- **Expressivity**：模型能不能表示复杂依赖。
- **Stability**：activation 和 gradient 是否保持在合理范围。
- **Efficiency**：训练和推理在真实硬件上是否高效。

今天的 notebook 只覆盖 tokenizer 的入口。Week 1 的 tiny MLP 则覆盖 training loop 的入口。


## 17. 和 Week 1 tiny MLP 的关系

你 Week 1 跑的 tiny MLP 不是语言模型，但它训练的是同一个核心闭环：

```text
input -> model -> prediction -> loss -> backward -> optimizer step
```

在语言模型里，它会变成：

```text
text -> tokenizer -> token ids -> Transformer -> logits -> cross-entropy loss -> backward -> optimizer step
```

所以第一周不是要你掌握完整 tokenization，也不是要你会写 Transformer。

第一周的目标是两件事：

1. 知道 CS336 为什么从 tokenization 开始。
2. 能解释一个 PyTorch training loop 里的 `forward -> loss -> backward -> optimizer.step`。


In [11]:
# 把两条线放在一起看：tokenizer 负责输入表示，training loop 负责参数更新。

pipeline = [
    "raw text",
    "tokenizer.encode(text)",
    "token ids",
    "embedding table",
    "Transformer",
    "logits",
    "cross entropy loss",
    "loss.backward()",
    "optimizer.step()",
]

for i, step in enumerate(pipeline, 1):
    print(f"{i}. {step}")


1. raw text
2. tokenizer.encode(text)
3. token ids
4. embedding table
5. Transformer
6. logits
7. cross entropy loss
8. loss.backward()
9. optimizer.step()


## 18. 今天的检查点

你看完这个 notebook 后，应该能用自己的话回答：

1. 为什么 CS336 强调 “from scratch”？
2. 为什么 frontier model 不能完整复现，但小模型仍然值得搭？
3. tokenizer 的 `encode` 和 `decode` 分别做什么？
4. character、byte、word tokenization 各有什么问题？
5. BPE 为什么是 bytes 和 words 之间的折中？
6. `compression ratio = bytes / tokens` 为什么和效率有关？
7. tokenization 和 Week 1 的 tiny MLP training loop 是怎么接上的？

如果这些能讲清楚，今天这一块就完成了。
